In [1]:
import pandas as pd
import numpy as np
import json

# Carrega os dados

In [16]:
# Lê os dados:
with open('../dados/limpos/usecases_current.json', 'r') as f:
    data = json.load(f)
usecases_raw = data['data']

In [17]:
# Seleciona casos considerados de uso de dados públicos:
usecases = list(filter(lambda uc: (uc['status_published'] == True) or ((fillnone(uc['comment'], '').find('Ocultado porque') == -1) and (fillnone(uc['comment'], '').find('não atendem aos nossos critérios') == -1)) , usecases_raw))

In [18]:
df = pd.DataFrame(usecases)

# Casos

In [19]:
def valor_ausente(valor):
    """Identifica nulos e coleções vazias ou compostas apenas por ausências."""
    if valor is None:
        return True
    if isinstance(valor, str):
        return valor.strip() == ""
    if isinstance(valor, (list, tuple, set)):
        return len(valor) == 0 or all(valor_ausente(item) for item in valor)
    if isinstance(valor, dict):
        return len(valor) == 0 or all(valor_ausente(item) for item in valor.values())

    resultado = pd.isna(valor)
    return bool(resultado) if isinstance(resultado, (bool, np.bool_)) else False


# Coleções parcialmente preenchidas são consideradas presentes nesta análise.
# A máscara permite contar e localizar ausências em cada coluna.
mascara_ausentes = pd.DataFrame({
    coluna: df[coluna].map(valor_ausente)
    for coluna in df.columns
})

resumo_ausentes = pd.DataFrame({
    "coluna": df.columns,
    "registros_faltantes": mascara_ausentes.sum().values,
    "percentual_faltante": (mascara_ausentes.mean() * 100).round(2).values,
    "registros_preenchidos": (~mascara_ausentes).sum().values,
}).sort_values("registros_faltantes", ascending=False, ignore_index=True)

display(resumo_ausentes)

# Exemplo para inspecionar os casos sem e-mail:
# df.loc[mascara_ausentes["email"], ["hash_id", "email"]]

,coluna,registros_faltantes,percentual_faltante,registros_preenchidos
0,authors_id,643,100.00,0
1,email,546,84.91,97
2,municipalities,544,84.60,99
3,fed_units,393,61.12,250
4,modified_date,174,27.06,469
5,countries,109,16.95,534
6,countries_es,109,16.95,534
7,pub_date,35,5.44,608
8,comment,12,1.87,631
9,geo_level,6,0.93,637


# Datasets

In [7]:
def parse_json(valor):
    """Normalizes the contents of the datasets column into a list of dicts.
    Accepts: a list/tuple/ndarray of dicts, a single dict, a JSON string, 
    a string in Python literal format, and null/empty values (returns []).
    """
    if isinstance(valor, dict):
        return [valor]
    if isinstance(valor, (list, tuple)):
        return list(valor)
    if hasattr(valor, "tolist"):        
        return valor.tolist()
    if valor is None or pd.isna(valor):
        return []

    # String
    texto = str(valor).strip()
    if texto in ("", "[]", "nan", "None"):
        return []
    try:
        resultado = json.loads(texto)
    except json.JSONDecodeError:
        resultado = ast.literal_eval(texto)

    return [resultado] if isinstance(resultado, dict) else list(resultado)

In [8]:
# Converte a coluna "datasets" em um dataframe
registros = []

for hash_id, valor in zip(df["hash_id"], df["datasets"]):
    for item in parse_json(valor):
        registros.append({"hash_id": hash_id, **item})

df_datasets = pd.DataFrame(registros)

assert df_datasets["hash_id"].isin(df["hash_id"]).all()

In [9]:
# Analisa dados faltantes coluna a coluna em df_datasets.
mascara_ausentes_datasets = pd.DataFrame({
    coluna: df_datasets[coluna].map(valor_ausente)
    for coluna in df_datasets.columns
})

resumo_ausentes_datasets = pd.DataFrame({
    "coluna": df_datasets.columns,
    "registros_faltantes": mascara_ausentes_datasets.sum().values,
    "percentual_faltante": (mascara_ausentes_datasets.mean() * 100).round(2).values,
    "registros_preenchidos": (~mascara_ausentes_datasets).sum().values,
}).sort_values("registros_faltantes", ascending=False, ignore_index=True)

display(resumo_ausentes_datasets)

# Exemplo para consultar os datasets sem licença:
# df_datasets.loc[mascara_ausentes_datasets["data_license"]]

,coluna,registros_faltantes,percentual_faltante,registros_preenchidos
0,data_periodical,70,3.63,1857
1,data_license,13,0.67,1914
2,data_format,11,0.57,1916
3,data_name,5,0.26,1922
4,data_url,3,0.16,1924
5,data_institution,1,0.05,1926
6,hash_id,0,0.00,1927
